# MOSAIC · HCMI four-setting OS prediction

Compare clinical-only, clinical + pathology, clinical + TP53 SNV, and all three on identical eligible patients. CNV remains unknown. Max pooling is within slides; patient risks are means across slides. No risk-direction grouping or gene weights are used.

Run cells in order. Edit the configuration cell before inference. Install `requirements-notebooks.txt` in this notebook kernel environment first.

## Download the example data

Get [`example_hcmi_feats/`](https://drive.google.com/drive/folders/1-8nydCaU-2qJdP3hLFM1GXUt2G3w_EZ-) and `hcmi_dataset.json` from the [MOSAIC Google Drive folder](https://drive.google.com/drive/folders/1IUbXoDB1KSfdNoSZMUHderIqItNFMSA1?usp=drive_link). Download and extract any ZIP archives, then place the feature folder and JSON directly under the repository's `data/` directory:

```text
data/
├── hcmi_dataset.json
└── example_hcmi_feats/
    └── *.h5
```

Keep `.h5` filenames unchanged and avoid an extra nested feature folder. You can use another location by editing `FEATURES` and `METADATA` below. Run the validation cell before inference to check the cohort.


In [ ]:
import copy
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Jupyter can start in the repository root or in notebooks/.
ROOT = Path.cwd()
if not (ROOT / "examples").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "examples" / "reproduce_hcmi_os.py").is_file():
    raise FileNotFoundError("Start Jupyter from the MOSAIC repository or notebooks/ directory.")
sys.path.insert(0, str(ROOT))
from examples import reproduce_hcmi_os as workflow


## 1 · Configure paths

Use `Astaxanthin/MOSAIC` or a downloaded MOSAIC directory. KEEP is the patch encoder, not the model to load here. CPU is a portable default; choose CUDA if your PyTorch installation supports it.

In [ ]:
MODEL = "Astaxanthin/MOSAIC"  # Or a local model folder
FEATURES = ROOT / "data/example_hcmi_feats"
METADATA = ROOT / "data/hcmi_dataset.json"
OUTPUT = ROOT / "outputs/hcmi_os"
DEVICE = "cpu"
REVISION = "main"
BOOTSTRAP = 1000


## 2 · Validate the cohort

This step checks metadata and feature availability without loading the model or running inference.

In [ ]:
cli = ["--no-interactive", "--model", MODEL, "--revision", REVISION,
       "--features", str(FEATURES), "--data", str(METADATA),
       "--output", str(OUTPUT), "--device", DEVICE,
       "--bootstrap", str(BOOTSTRAP)]

args = workflow.parse_args(cli)
check = copy.copy(args)
check.validate_only = True
display(workflow.run(check))


## 3 · Run the reproduction

The first run extracts WSI features and may take several minutes on CPU. Progress is printed. Completed slides are cached; rerunning with the same configuration resumes them. Select a new output directory if you change the model or evaluation settings.

In [ ]:
result = workflow.run(args)

## 4 · Inspect results

In [ ]:
metrics = pd.read_csv(Path(args.output) / "metrics.csv")
patients = pd.read_csv(Path(args.output) / "patient_predictions.csv")
display(metrics)
display(patients.head(10))

names = {"clinical": "Clinical", "clinical_vision": "Clinical + pathology",
         "clinical_genomics": "Clinical + TP53 SNV",
         "clinical_vision_genomics": "Clinical + pathology + TP53 SNV"}
for project, rows in metrics.groupby("project", sort=False):
    rows = rows.set_index("setting").loc[list(names)]
    y = rows["cindex"]
    errors = [y - rows["ci_lower"], rows["ci_upper"] - y]
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.errorbar(y, range(4), xerr=errors, fmt="o", color="#28658f", capsize=3)
    ax.set(yticks=range(4), yticklabels=list(names.values()), xlabel="OS C-index (bootstrap 95% CI)", xlim=(0, 1), title=project)
    ax.invert_yaxis()
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    plt.show()
